<a href="https://colab.research.google.com/github/siddharthshuklacs24/sem-5-AI/blob/main/1BF24CS289_Week10_AlphaBeta_Pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Alpha-Beta Pruning on a Game Tree
- The root is a MAX node; levels alternate MIN / MAX.
- The tree is nested lists; numbers are leaf utility values.
"""

import math

# Root (MAX) -> MIN nodes A, B -> MAX nodes -> leaves
TREE = [
    [[10, 9], [14, 18]],   # A (MIN): A1 (MAX), A2 (MAX)
    [[5, 4], [50, 3]],     # B (MIN): B1 (MAX), B2 (MAX)
]

pruned_log = []   # (node name, leaf values) for every pruned subtree

def leaves_of(node):
    if not isinstance(node, list):
        return [node]
    return [v for child in node for v in leaves_of(child)]

def alpha_beta(node, name, alpha, beta, is_max, depth=0):
    """Return (value, path) for node using alpha-beta pruning."""
    indent = "    " * depth
    kind = "MAX" if is_max else "MIN"

    if not isinstance(node, list):           # leaf node
        print(f"{indent}Leaf {name} = {node}")
        return node, [name]

    print(f"{indent}Visit {name} ({kind})  alpha={alpha}, beta={beta}")
    value = -math.inf if is_max else math.inf
    best_path = []

    for i, child in enumerate(node):
        child_name = f"{name}.{i + 1}"
        child_value, child_path = alpha_beta(child, child_name, alpha, beta,
                                             not is_max, depth + 1)

        if (is_max and child_value > value) or (not is_max and child_value < value):
            value = child_value
            best_path = [name] + child_path

        if is_max:
            alpha = max(alpha, value)
        else:
            beta = min(beta, value)

        if beta <= alpha:                     # cutoff: skip remaining children
            for j in range(i + 1, len(node)):
                skipped = f"{name}.{j + 1}"
                leaves = leaves_of(node[j])
                pruned_log.append((skipped, leaves))
                print(f"{indent}  PRUNE {skipped} leaves={leaves} "
                      f"(alpha={alpha}, beta={beta})")
            break

    print(f"{indent}Return {name} ({kind}) = {value}")
    return value, best_path

if __name__ == "__main__":
    value, path = alpha_beta(TREE, "root", -math.inf, math.inf, is_max=True)

    print("\nValue of root (MAX):", value)
    print("Path to root:", " -> ".join(path[:-1]), f"-> leaf {path[-1]}")
    print("Pruned subtrees:")
    for name, leaves in pruned_log:
        print(f"  {name} with leaves {leaves}")

Visit root (MAX)  alpha=-inf, beta=inf
    Visit root.1 (MIN)  alpha=-inf, beta=inf
        Visit root.1.1 (MAX)  alpha=-inf, beta=inf
            Leaf root.1.1.1 = 10
            Leaf root.1.1.2 = 9
        Return root.1.1 (MAX) = 10
        Visit root.1.2 (MAX)  alpha=-inf, beta=10
            Leaf root.1.2.1 = 14
          PRUNE root.1.2.2 leaves=[18] (alpha=14, beta=10)
        Return root.1.2 (MAX) = 14
    Return root.1 (MIN) = 10
    Visit root.2 (MIN)  alpha=10, beta=inf
        Visit root.2.1 (MAX)  alpha=10, beta=inf
            Leaf root.2.1.1 = 5
            Leaf root.2.1.2 = 4
        Return root.2.1 (MAX) = 5
      PRUNE root.2.2 leaves=[50, 3] (alpha=10, beta=5)
    Return root.2 (MIN) = 5
Return root (MAX) = 10

Value of root (MAX): 10
Path to root: root -> root.1 -> root.1.1 -> leaf root.1.1.1
Pruned subtrees:
  root.1.2.2 with leaves [18]
  root.2.2 with leaves [50, 3]
